# ДеревоВартовий — NDVI pipeline (Colab)

Генерує артефакти для `5–8` подій рубок у форматі, готовому для Next.js/Vercel фронтенду.

**Вхід:** список AOI (координати центру + bbox + дві дати).

**Вихід на кожну подію** в `output/<event_id>/`:
- `before.png` — RGB-композит «до»
- `after.png` — RGB-композит «після»
- `ndvi_diff.png` — маска змін (напівпрозора, червоним)
- `event.json` — метадані для фронтенду

Плюс загальний `events.json` зі списком усіх подій — його кладете в `public/data/` Next.js проєкту.

---

## 0. Підготовка

In [ ]:
!pip install -q requests numpy pillow matplotlib tifffile

In [ ]:
import os, json, io, zipfile, time
from pathlib import Path
from datetime import datetime
import requests
import numpy as np
from PIL import Image
import tifffile
import matplotlib.pyplot as plt

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. OAuth credentials

Створіть OAuth client у Sentinel Hub Dashboard (`User Settings → OAuth clients → Create`) і вставте `client_id` / `client_secret` нижче. Можна через Colab Secrets (`🔑` у лівому сайдбарі), але для простоти — прямо тут.

Токен живе ~1 годину, ноутбук його оновлює автоматично.

In [ ]:
CLIENT_ID = 'sh-a510f8c8-264b-47fc-ab3f-ce88a2a2316c'
CLIENT_SECRET = 'quXCwknu0o5Ri2igvhBHG1LZrPIJfDM2'

TOKEN_URL = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
PROCESS_URL = 'https://sh.dataspace.copernicus.eu/api/v1/process'

def get_token():
    r = requests.post(TOKEN_URL, data={
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    })
    r.raise_for_status()
    return r.json()['access_token']

TOKEN = get_token()
print('OAuth OK, token length:', len(TOKEN))

## 2. Список AOI

Це і є ваш «датасет» для демки. Для кожної локації — `bbox` у WGS84 `[west, south, east, north]` і дві дати.

**Як підібрати локації:** відкрийте [Copernicus Browser](https://browser.dataspace.copernicus.eu/), знайдіть місце в Карпатах, де візуально видно свіжу рубку (світла пляма серед темного лісу), і запишіть bbox. Дати — 2 місяці до рубки і 2 місяці після, літні (червень–вересень), щоб не було снігу.

Розмір bbox: ~`0.02°` × `0.02°` (приблизно 2×2 км) — добре для демки, не дуже великі файли.

In [ ]:
AOIS = [
    {
        'id': 'carpathians_01',
        'name': 'Ділянка №1, Івано-Франківська область',
        'region': 'Івано-Франківська',
        'bbox': [24.50, 48.25, 24.53, 48.28],
        'date_before': ('2024-06-01', '2024-06-30'),
        'date_after':  ('2024-08-15', '2024-09-15'),
    },
    {
        'id': 'carpathians_02',
        'name': 'Ділянка №2, Закарпатська область',
        'region': 'Закарпатська',
        'bbox': [23.80, 48.45, 23.83, 48.48],
        'date_before': ('2024-06-01', '2024-06-30'),
        'date_after':  ('2024-08-15', '2024-09-15'),
    },
    # Додайте ще 3–6 локацій
]

print(f'Всього AOI: {len(AOIS)}')

## 3. Evalscripts

Два скрипти: один повертає RGB для візуалізації, другий — NDVI як float32 GeoTIFF-ніби (насправді просто масив, ми приймаємо `TIFF` і парсимо).

In [ ]:
EVALSCRIPT_RGB = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ['B02','B03','B04','SCL'] }],
    output: { bands: 3, sampleType: 'AUTO' }
  };
}
function evaluatePixel(s) {
  // Трохи підтягнемо контраст: 2.5× стандартна практика для Sentinel-2 RGB
  return [2.5*s.B04, 2.5*s.B03, 2.5*s.B02];
}
"""

EVALSCRIPT_NDVI = """
//VERSION=3
function setup() {
  return {
    input: [{ bands: ['B04','B08','SCL'] }],
    output: { bands: 1, sampleType: 'FLOAT32' }
  };
}
function evaluatePixel(s) {
  // SCL: 3=cloud_shadow, 8=cloud_medium, 9=cloud_high, 10=thin_cirrus, 11=snow
  if ([3,8,9,10,11].indexOf(s.SCL) >= 0) return [NaN];
  let ndvi = (s.B08 - s.B04) / (s.B08 + s.B04 + 1e-9);
  return [ndvi];
}
"""

## 4. Запит до Process API

In [ ]:
def fetch_image(bbox, date_from, date_to, evalscript, response_format='image/png', width=512, height=512):
    global TOKEN
    body = {
        'input': {
            'bounds': {
                'bbox': bbox,
                'properties': {'crs': 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'},
            },
            'data': [{
                'type': 'sentinel-2-l2a',
                'dataFilter': {
                    'timeRange': {
                        'from': f'{date_from}T00:00:00Z',
                        'to':   f'{date_to}T23:59:59Z',
                    },
                    'maxCloudCoverage': 30,
                    'mosaickingOrder': 'leastCC',
                },
            }],
        },
        'output': {
            'width': width,
            'height': height,
            'responses': [{'identifier': 'default', 'format': {'type': response_format}}],
        },
        'evalscript': evalscript,
    }
    headers = {'Authorization': f'Bearer {TOKEN}'}
    r = requests.post(PROCESS_URL, json=body, headers=headers)
    if r.status_code == 401:
        TOKEN = get_token()
        headers = {'Authorization': f'Bearer {TOKEN}'}
        r = requests.post(PROCESS_URL, json=body, headers=headers)
    r.raise_for_status()
    return r.content

## 5. Парсинг NDVI і маска різниці

In [ ]:
def parse_float_tiff(tiff_bytes):
    """Читаємо однобандовий FLOAT32 TIFF через tifffile (Pillow ламається на FLOAT32)."""
    return tifffile.imread(io.BytesIO(tiff_bytes)).astype(np.float32)

def bbox_area_ha(bbox):
    """Груба оцінка площі bbox у гектарах (для ~48° широти)."""
    w, s, e, n = bbox
    lat_m = (n - s) * 111_000
    lon_m = (e - w) * 111_000 * np.cos(np.radians((n+s)/2))
    return (lat_m * lon_m) / 10_000

def make_diff_mask(ndvi_before, ndvi_after, threshold=0.2):
    """Маска пікселів зі значним NDVI drop. Повертає RGBA PNG bytes."""
    valid = ~(np.isnan(ndvi_before) | np.isnan(ndvi_after))
    drop = np.where(valid, ndvi_before - ndvi_after, 0)
    loss_mask = (drop > threshold) & valid

    h, w = loss_mask.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[loss_mask] = [255, 40, 40, 180]  # червоний з alpha

    buf = io.BytesIO()
    Image.fromarray(rgba, mode='RGBA').save(buf, format='PNG')
    stats = {
        'pixels_total': int(valid.sum()),
        'pixels_loss': int(loss_mask.sum()),
        'loss_fraction': float(loss_mask.sum() / max(valid.sum(), 1)),
        'ndvi_mean_before': float(np.nanmean(ndvi_before)),
        'ndvi_mean_after': float(np.nanmean(ndvi_after)),
        'ndvi_drop_mean': float(np.nanmean(drop[loss_mask])) if loss_mask.sum() > 0 else 0.0,
    }
    return buf.getvalue(), stats

## 6. Обробка всіх AOI

In [ ]:
events = []

for aoi in AOIS:
    print(f"\n→ {aoi['id']}: {aoi['name']}")
    event_dir = OUTPUT_DIR / aoi['id']
    event_dir.mkdir(exist_ok=True)

    try:
        # RGB «до» і «після»
        rgb_before = fetch_image(aoi['bbox'], *aoi['date_before'], EVALSCRIPT_RGB, 'image/png')
        (event_dir / 'before.png').write_bytes(rgb_before)
        print('  RGB before: OK')

        rgb_after = fetch_image(aoi['bbox'], *aoi['date_after'], EVALSCRIPT_RGB, 'image/png')
        (event_dir / 'after.png').write_bytes(rgb_after)
        print('  RGB after: OK')

        # NDVI як float TIFF
        ndvi_before_tiff = fetch_image(aoi['bbox'], *aoi['date_before'], EVALSCRIPT_NDVI, 'image/tiff')
        ndvi_after_tiff  = fetch_image(aoi['bbox'], *aoi['date_after'],  EVALSCRIPT_NDVI, 'image/tiff')
        ndvi_before = parse_float_tiff(ndvi_before_tiff)
        ndvi_after  = parse_float_tiff(ndvi_after_tiff)
        print(f'  NDVI: before mean={np.nanmean(ndvi_before):.3f}, after mean={np.nanmean(ndvi_after):.3f}')

        # Маска різниці
        mask_png, stats = make_diff_mask(ndvi_before, ndvi_after, threshold=0.2)
        (event_dir / 'ndvi_diff.png').write_bytes(mask_png)

        # Груба оцінка площі втрат у гектарах
        total_ha = bbox_area_ha(aoi['bbox'])
        loss_ha = total_ha * stats['loss_fraction']

        # Confidence — простий ad-hoc score
        confidence = min(1.0, stats['loss_fraction'] * 5 + stats['ndvi_drop_mean'])

        event = {
            'id': aoi['id'],
            'name': aoi['name'],
            'region': aoi['region'],
            'bbox': aoi['bbox'],
            'center': [(aoi['bbox'][0]+aoi['bbox'][2])/2, (aoi['bbox'][1]+aoi['bbox'][3])/2],
            'date_before': aoi['date_before'][1],
            'date_after': aoi['date_after'][0],
            'area_total_ha': round(total_ha, 1),
            'area_loss_ha': round(loss_ha, 2),
            'ndvi_drop_mean': round(stats['ndvi_drop_mean'], 3),
            'ndvi_before_mean': round(stats['ndvi_mean_before'], 3),
            'ndvi_after_mean': round(stats['ndvi_mean_after'], 3),
            'confidence': round(confidence, 2),
            'assets': {
                'before': f'/data/events/{aoi["id"]}/before.png',
                'after':  f'/data/events/{aoi["id"]}/after.png',
                'mask':   f'/data/events/{aoi["id"]}/ndvi_diff.png',
            },
        }
        (event_dir / 'event.json').write_text(json.dumps(event, ensure_ascii=False, indent=2))
        events.append(event)
        print(f'  ✓ loss: {loss_ha:.2f} га, confidence: {confidence:.2f}')
    except Exception as e:
        print(f'  ✗ ПОМИЛКА: {e}')

# Загальний events.json
(OUTPUT_DIR / 'events.json').write_text(json.dumps({
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'count': len(events),
    'events': events,
}, ensure_ascii=False, indent=2))

print(f'\n=== Готово: {len(events)} подій ===')

## 7. Швидкий перегляд

In [ ]:
for event in events:
    eid = event['id']
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(Image.open(OUTPUT_DIR / eid / 'before.png')); axes[0].set_title(f"{eid} — до"); axes[0].axis('off')
    axes[1].imshow(Image.open(OUTPUT_DIR / eid / 'after.png'));  axes[1].set_title('після'); axes[1].axis('off')
    axes[2].imshow(Image.open(OUTPUT_DIR / eid / 'after.png'))
    axes[2].imshow(Image.open(OUTPUT_DIR / eid / 'ndvi_diff.png'), alpha=0.7)
    axes[2].set_title(f'маска (втрата {event["area_loss_ha"]} га)'); axes[2].axis('off')
    plt.tight_layout(); plt.show()

## 8. Експорт у ZIP для Next.js

Розпакуйте в `public/data/` вашого Next.js проєкту. `events.json` буде доступний за `/data/events.json`, PNG-и за шляхами з `event.assets`.

In [ ]:
zip_path = 'treeguardian_data.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_DIR / 'events.json', 'events.json')
    for event in events:
        eid = event['id']
        for fname in ['before.png', 'after.png', 'ndvi_diff.png', 'event.json']:
            zf.write(OUTPUT_DIR / eid / fname, f'events/{eid}/{fname}')

print(f'Архів: {zip_path}')
from google.colab import files
files.download(zip_path)